# 05 · E2 (árboles) y E3 (variables territoriales)
Lógica en `src/modelos_arboles.py` y `src/territorio.py`. Mismo protocolo que E0 y E1 (`src/protocolo.py`): rolling-origin con reentrenamiento mensual sobre **2024 (validación)**. **2025 no se toca.**

- **E2:** LightGBM Poisson y Random Forest con el historial local (`lag_1`, `lag_12`, `ma_3`, `ma_6`, `ma_12`) y el mes. LightGBM con *offset* `log(ma_12 + 0,01)` es el principal.
- **E3:** E2 + variables fijas de la zona (posición, tamaño, distrito). Es una **ablación**: ¿aporta algo la heterogeneidad espacial fija?

**Cautela:** los hiperparámetros y la ventana de entrenamiento se eligieron mirando la validación 2024 (132 configuraciones en la búsqueda de E2). Las cifras de validación pueden ser algo optimistas.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from baselines import predecir_e0
from features import anadir_rezagos
from modelos_arboles import lightgbm_poisson, random_forest
from modelos_conteo import glm_poisson
from protocolo import MESES_VALIDACION, evaluar_rolling, resumir
from territorio import GRUPOS, con_territorio, tabla_territorial

PROCESSED = ROOT / "data" / "processed"
RESOLUCIONES = ["1km", "2km", "distritos"]
paneles = {n: pd.read_parquet(PROCESSED / f"panel_{n}.parquet") for n in RESOLUCIONES}
datos = {n: anadir_rezagos(p, lags=(1, 12), ventanas=(3, 6, 12)) for n, p in paneles.items()}

## Mejores hiperparámetros de LightGBM (elegidos en validación 2024)
Búsqueda acotada de 132 configuraciones (44 por resolución). Criterio fijado antes de ver los resultados: **mínima devianza de Poisson en 2024**.

In [ ]:
PARAMS = {
    "1km": dict(num_leaves=15, n_estimators=150, min_child_samples=50),
    "2km": dict(num_leaves=8, n_estimators=150, min_child_samples=50),
    "distritos": dict(num_leaves=15, n_estimators=600, min_child_samples=50),
}
pd.DataFrame(PARAMS).T

## E2 frente a E0 y E1
Misma validación, mismas filas y mismas métricas. Se muestran el baseline a batir (`media_12_estacional`), E1, LightGBM sin y con offset y un Random Forest con los valores por defecto del módulo (200 árboles, hoja mínima 20). El Random Forest afinado en la búsqueda de 132 configuraciones queda algo mejor (1 km: MAE 0,793 / devianza 0,581; 2 km: 1,516 / 0,698; distritos: 9,158 / 1,753), pero sigue por detrás de LightGBM con offset.

In [ ]:
def tabla_e2(res):
    d = datos[res]
    filas = {"media_12_estacional": resumir(predecir_e0(paneles[res], MESES_VALIDACION)["media_12_estacional"])}
    g, gc = glm_poisson()
    filas["E1 GLM (offset)"] = resumir(evaluar_rolling(d, MESES_VALIDACION, g, gc))
    for nombre, off in [("LightGBM", False), ("LightGBM + offset", True)]:
        f, c = lightgbm_poisson(PARAMS[res], con_offset=off)
        filas[nombre] = resumir(evaluar_rolling(d, MESES_VALIDACION, f, c))
    f, c = random_forest()
    filas["Random Forest (por defecto)"] = resumir(evaluar_rolling(d, MESES_VALIDACION, f, c))
    return pd.DataFrame(filas).T.round(3)

e2 = {r: tabla_e2(r) for r in RESOLUCIONES}
e2["1km"]

In [ ]:
e2["2km"]

In [ ]:
e2["distritos"]

**Lectura.** E2 no mejora a E1 ni al baseline en MAE. LightGBM con offset reproduce a E1 en devianza a 1 km y 2 km, y es peor en distritos. Sin offset es inestable. Los árboles no encuentran señal no lineal extra en el historial local: el nivel de la celda y el mes explican casi todo lo explicable con estos datos.

## ¿Entrenar sin los años de COVID también en E2?
Se comprueba con LightGBM + offset (mejor configuración de cada resolución) y la validación 2024. `entreno_desde=None` usa todo.

In [ ]:
filas = []
for res in RESOLUCIONES:
    for nombre, desde in [("todo", None), ("2021-01", pd.Timestamp("2021-01-01")), ("2022-01", pd.Timestamp("2022-01-01"))]:
        f, c = lightgbm_poisson(PARAMS[res], con_offset=True, entreno_desde=desde)
        r = resumir(evaluar_rolling(datos[res], MESES_VALIDACION, f, c))
        filas.append({"resolucion": res, "entrena desde": nombre, "MAE": r["MAE"], "devianza": r["devianza_poisson"]})
pd.DataFrame(filas).round(3)

**Lectura.** Incluir 2020–2021 empeora E2 en las tres resoluciones (en distritos, devianza 2,03 frente a 1,59). Se mantiene **2022-01**, igual que E1, así que la comparación E1 frente a E2 usa las mismas filas de entrenamiento.

## E3: variables territoriales
`src/territorio.py` calcula, solo con los contornos oficiales, cinco variables fijas por zona: `coord_x`, `coord_y` (centroide en km), `area_km2`, `frac_dentro` y `distrito` (el de mayor solape con la celda). No dependen de accidentes ni del tiempo.

In [ ]:
tabla_territorial("1km").describe().round(2)

### Ablación (LightGBM + offset)
Cada grupo por separado y todos juntos, con los mismos hiperparámetros que E2.

In [ ]:
COMBOS = {
    "E2 (sin territorio)": (),
    "+ posición": GRUPOS["posicion"],
    "+ tamaño": GRUPOS["tamano"],
    "+ distrito": GRUPOS["distrito"],
    "+ todo": sum(GRUPOS.values(), ()),
}
filas = []
for res in RESOLUCIONES:
    d = anadir_rezagos(con_territorio(paneles[res], res), lags=(1, 12), ventanas=(3, 6, 12))
    for nombre, extra in COMBOS.items():
        f, c = lightgbm_poisson(PARAMS[res], con_offset=True, extra=extra)
        r = resumir(evaluar_rolling(d, MESES_VALIDACION, f, c))
        filas.append({"resolucion": res, "variables": nombre, "MAE": r["MAE"],
                      "devianza": r["devianza_poisson"], "Capture@10%": r["capture@10%"]})
ablacion = pd.DataFrame(filas).round(3)
ablacion.pivot(index="variables", columns="resolucion", values="devianza")[RESOLUCIONES]

In [ ]:
ablacion

**Lectura.** Las variables territoriales no aportan: a 1 km y 2 km la devianza cambia unas milésimas (dentro del ruido) y en distritos empeora, porque con 21 zonas solo sirven para memorizar. MAE y Capture@10 % no cambian. El offset `ma_12` ya recoge el nivel fijo de cada zona.

## Conclusión de la Etapa 4
- E2 y E3 no superan a E1 ni al baseline en MAE; en devianza, E1 y E2 mejoran al baseline a 1 km y 2 km.
- La heterogeneidad espacial fija no añade información sobre el historial local.
- Lo que queda por explorar es la señal de **zonas vecinas** (H2, Etapa 5).